# <font color='orange'>TME ROBOTIQUE ET APPRENTISSAGE</font>
# <font color='orange'>2. Robotique Evolutionnaire</font>

# <font color="green">Version ETUDIANTS·ES 2025-2026</font>

*contact: NB@SU*

*mise à jour: 16/3/2026*


Ce notebook doit être exécuté dans [Google Colab](colab.research.google.com/) ou dans Jupyter.

A la fin du TME, vous devez déposer votre travail sur Moodle:
* déposer votre notebook, avec le nom de fichier *obligatoirement* au format suivant: **RA_NOM1_NOM2.ipynb**
* toutes les cellules doivent être exécutées avec les graphes visibles
* un **bref** commentaire lorsque c'est demandé. Pour toutes les questions, une à deux phrases suffisent

*Le sujet est à faire en binome.*

# COMPLETEZ LES CHAMPS CI-DESSOUS AVEC NOM/PRENOM/CARTE_ETU:

* Étudiant 1: **_Nom_ _Prénom_ _noCarteEtudiant_**
* Étudiant 2: **_Nom_ _Prénom_ _noCarteEtudiant_**



# Simple Robot Simulator



*   simulate renvoie des stats sur le comportement du robot
  * distance parcourue a chaque pas
  * exploration grille (si utilise trace systématiquement)
*   ...



---
---
---

# <font color='orange'>PREAMBULE: initialisation</font>

# Simulateur robot

Il n'est pas nécessaire de regarder le code de ces cellules, elles sont utiles pour les fonctions utilisées dans les parties suivantes.

Remarque: *il suffit d'exécuter ces cellules une seule fois par instance de noyau.*

In [ ]:
from datetime import datetime
from datetime import date
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
import numpy as np
import math
import random
import sys

!pip install cma
import cma

#### #### ####

print("\n",date.today(), datetime.now().strftime("%H:%M:%S"),"GMT") # timestamp is greenwich time
print("OK.")


 2024-03-14 09:50:41 GMT
OK.


In [ ]:
#############################
### 0. Experimental setup ###
#############################

max_iterations = 5001

absolute_orientation = 0 #-90  # degrees
translation_per_step = 1.0 #1.0 # 6.0  # in [0,1]
max_translation_per_step = 1.0 # >0. pixel per move.
rotation_per_step = 0.0 # in [-1,1]
max_rotation_per_step = 3.0 # >0, in degrees

arena_size = 100 # 50
sensor_length = 10
nb_sensors = 8

particle_box = 3 # bounding box (square)

particle = np.zeros((particle_box, particle_box), dtype=int)

# Drawing a circle in the particle array
#particle_radius_real = particle_box / 2 - 0.5  # Radius of the circle in the particle
#center_particle_real = (particle_box / 2 - 0.5, particle_box / 2 - 0.5)  # Real center of the particle
particle_radius_real = particle_box / 2.0  # Radius of the circle in the particle
center_particle_real = (particle_box / 2 - 0.5, particle_box / 2 - 0.5)  # Real center of the particle

####################
### 1. Structure ###
####################

def init(): # perform only once
  global particle,center_particle_real,particle_radius_real

  for i in range(particle.shape[0]):
      for j in range(particle.shape[1]):
          # Check if the cell falls within the circle radius
          if ((i - center_particle_real[0])**2 + (j - center_particle_real[1])**2) <= particle_radius_real**2:
              particle[i, j] = 2
          else:
            particle[i, j] = 3

  init_arena()
  init_trace()

def init_arena():
  global arena
  arena = np.zeros((arena_size, arena_size), dtype=int)

def init_trace():
  global trace
  trace = np.zeros((arena_size, arena_size), dtype=int)

def clear_arena(): # clear content, except for wall
  global arena
  #arena[:] = 0
  arena[arena != 1] = 0

def clear_trace(): # clear content, except for wall
  global trace
  #trace[:] = 0
  trace[trace != 1] = 0

def draw_line(x1, y1, x2, y2, color): # do not draw on wall
    global arena
    num_points = max(abs(x2 - x1), abs(y2 - y1)) + 1
    x_points = np.linspace(x1, x2, num_points, dtype=int)
    y_points = np.linspace(y1, y2, num_points, dtype=int)
    # shorte, not border-protected:
    # arena[y_points, x_points] = color
    # longer, border-protected:
    for i in range(len(x_points)):
      if y_points[i] >= 0 and y_points[i] < arena.shape[0] and x_points[i] >= 0 and x_points[i] < arena.shape[1]:
        if arena[y_points[i],x_points[i]] != 1:
          arena[y_points[i],x_points[i]] = color

def cast_sensor(x1, y1, x2, y2, color=6):
    global arena, display_cast
    num_points = max(abs(x2 - x1), abs(y2 - y1)) + 1
    x_points = np.linspace(x1, x2, num_points, dtype=int)
    y_points = np.linspace(y1, y2, num_points, dtype=int)
    max_distance = math.sqrt( (x_points[0]-x_points[-1])**2 + (y_points[0]-y_points[-1])**2 ) - particle_radius_real
    for i in range(len(x_points)):
      if y_points[i] >= 0 and y_points[i] < arena.shape[0] and x_points[i] >= 0 and x_points[i] < arena.shape[1]:
        if arena[y_points[i],x_points[i]] == 1 or arena[y_points[i],x_points[i]] == 2  or arena[y_points[i],x_points[i]] == 4:
          if verbose:
            print ("DETECT OBJECT at",i)
          return ( math.sqrt( (x_points[0]-x_points[i])**2 + (y_points[0]-y_points[i])**2 ) - particle_radius_real ) / max_distance # distance to obstacle, normalize in [0,1]
        elif display_cast == True:
          arena[y_points[i],x_points[i]] = color
    return 1.0

absolute_orientation = 0.0  # in degrees
translation_per_step = 0.0  # in units
rotation_per_step = 0.0  # in degrees

############################
### 2. General functions ###
############################

def create_wall(x1, y1, x2, y2):
    arena[max(y1, 0):min(y2, arena_size), max(x1, 0):min(x2, arena_size)] = 1
    if display_trace:
      trace[max(y1, 0):min(y2, arena_size), max(x1, 0):min(x2, arena_size)] = 1

def get_sensors(x, y, absolute_orientation): # 8 sensors belt
    global sensor_length, nb_sensors

    source_x, source_y = int(x+particle_radius_real-1), int(y+particle_radius_real-1)

    sensor_values = []
    for i in range(nb_sensors):
      offset_x = math.cos(math.radians(absolute_orientation+i*360.0/nb_sensors)) * sensor_length
      offset_y = math.sin(math.radians(absolute_orientation+i*360.0/nb_sensors)) * sensor_length
      target_x = int(source_x + offset_x + 0.5)
      target_y = int(source_y - offset_y + 0.5)
      sensor_values.append( cast_sensor(int(source_x),int(source_y),target_x,target_y,6) )

    return sensor_values

def clean_sensors(x, y, absolute_orientation): # 8 sensors belt
    global sensor_length, nb_sensors

    if display_cast == False:
      return

    source_x, source_y = int(x+particle_radius_real-1), int(y+particle_radius_real-1)

    sensor_length = 10
    nb_sensors = 8

    sensor_values = []
    for i in range(nb_sensors):
      offset_x = math.cos(math.radians(absolute_orientation+i*360.0/nb_sensors)) * sensor_length
      offset_y = math.sin(math.radians(absolute_orientation+i*360.0/nb_sensors)) * sensor_length
      target_x = int(source_x + offset_x + 0.5)
      target_y = int(source_y - offset_y + 0.5)
      draw_line(int(source_x),int(source_y),target_x,target_y,0)

    return sensor_values

def place_particle(x, y,absolute_orientation):
    global arena, particle
    x, y = int(x), int(y)
    arena_slice = arena[y:y+particle_box, x:x+particle_box]

    # Check for overlap between particle and non-zero cells in the arena
    collision = np.logical_and(arena_slice == 1, particle == 2)
    if np.any(collision):
        collision_array = np.where(collision, 3, particle)
        return collision_array

    #arena_slice[ int(math.cos(absolute_orientation)*particle_box/2) , int(math.sin(absolute_orientation)*particle_box/2) ] = 3 # display particle front
    #arena[y:y+particle_box, x:x+particle_box] += np.where(arena_slice == 0, particle, 0)

    for i in range(particle_box):
      for j in range(particle_box):
        if arena[y+i, x+j] == 0 or arena[y+i, x+j] == 4 or arena[y+i, x+j] == 6:
          arena[y+i, x+j] = 2

    trace[int(y + particle_box / 2.),int(x + particle_box / 2.)] = 2

    if verbose:
      print("location",x+particle_box/2.0,y+particle_box/2.0)
      print("absolute_orientation", absolute_orientation)

    offset_x = math.cos(math.radians(absolute_orientation)) * particle_box / 2.0 * .999
    offset_y = math.sin(math.radians(absolute_orientation)) * particle_box / 2.0 * .999
    x_head = int(x + particle_box / 2. + offset_x )
    y_head = int(y + particle_box / 2. - offset_y )
    if verbose:
      print ("y_head, x_head",y_head, x_head)

    arena[y_head, x_head] = 4  # Display particle front

    return None

def erase_particle(x, y):
    global arena, particle
    x, y = int(x), int(y)
    #arena[y:y+particle_box, x:x+particle_box] -= particle * (arena[y:y+particle_box, x:x+particle_box] != 0)
    for i in range(particle_box):
      for j in range(particle_box):
        if arena[y+i, x+j] == 2 or arena[y+i, x+j] == 4:
          arena[y+i, x+j] = 0

def draw(arena,double_size=False):

    if double_size == True:
      doubled_figsize = (12.8, 9.6)
      plt.figure(figsize=doubled_figsize)

    cmap = colors.ListedColormap(['white', 'black', 'grey', 'red', 'orange', 'green'])
    bounds = [0, 1, 2, 3, 4, 5, 6]
    norm = colors.BoundaryNorm(bounds, cmap.N)

    plt.imshow(arena, cmap=cmap, norm=norm)
    plt.colorbar(ticks=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5], format=plt.FuncFormatter(lambda x, _: {0.5: '0 - nothing', 1.5: '1 - obstacle', 2.5: '2 - robot', 3.5: '3 - collision', 4.5: '4 - robot front', 5.5: '5 - sensor ray'}[x]))
    plt.show()

def show_arena():
  draw(arena,double_size=False)

def show_trace():
  draw(trace,double_size=False)


####################################
### 3. PARTICLE UPDATE functions ###
####################################

def update_particle_position(x, y, absolute_orientation, translation_per_step, rotation_per_step, iteration, noiseFlag = False):
  '''
  Function called to update particle position (and orientation). It should merge the desired control command from the user (translation_per_step and rotation_per_step) and the physics of environment and particle.
  Below is a simple example where desired translation and rotation per step are applied directly (ideal: no friction, unbounded bang-bang controller)
  Add whatever you need as persistent internal data.

  # x, y: position of the particle (real values)
  # absolute_orientation: particle absolute orientation wrt an external referential (0° points rightward, trigonometric, use degree)
  # translation_per_step, rotation_per_step: *desired* value by the controller/user. I.e.: values are applied if no friction, no inertia, no collision (see example below)
  # iteration: simulator iteration number (may be ignored)
  '''

  new_orientation = absolute_orientation + rotation_per_step

  # Sinusoidal noise for rotation (optional)
  if noiseFlag == True:
    noise = 10 * math.sin(2 * math.pi * iteration / 5)
    new_orientation += noise

  # Convert orientation to radians for calculation
  orientation_radians = math.radians(new_orientation)

  new_x = x + translation_per_step * math.cos(orientation_radians)
  new_y = y - translation_per_step * math.sin(orientation_radians)  # Minus sign due to y-axis inversion in image coordinates

  return new_x, new_y, new_orientation


def update_particle_dynamics(x, y, absolute_orientation, translation_per_step, rotation_per_step, iteration, collision_array ):
  '''
  Function called to udpate internal data (to be defined here) in case of a collision event.
  This function does not move the robot. Updating the absolute_orientation is possible, as well as (to be defined) internal data structure (e.g. encoding the effect of inertia)
  Note that when a particle experience a collision, the collision is recorded (see collision_array) and move is cancelled.
  The goal of this function is to compute/update all necessary information to call later the update_particle_position function.
  Add whatever you need as persistent internal data.

  # x, y: position of the particle (real values)
  # absolute_orientation: particle absolute orientation wrt an external referential (0° points rightward, trigonometric, use degree)
  # translation_per_step, rotation_per_step: *desired* value by the controller/user. I.e.: values are applied if no friction, no inertia, no collision (see example below)
  # iteration: simulator iteration number (may be ignored)
  # iteration: simulator iteration number (may be ignored)
  # collision_array: integer array showing the particle state within its bounding box. 0 for nothing, 1 for particle elements, 2 for particle elements experiencing a collision event

  Testing: essaye avec translation_per_step=1.0, puis avec translation_per_step=6.0 (cas intéressant du a la faible épaisseur du mur)
  '''

  # do nothing for now.

  return absolute_orientation


#######################
### 4. Main loop    ###
#######################

# parameters
  # initial x and y position and orientation
  # step controller function which returns translation/rotation (in [-1,1]) from sensors (in [0,1])
# returns:
  # sum of actual translations in [0,1]. Normalized over time and max translation.
  # sum of rotation, absolute value normalized value in [0,1]. Normalized over time and max rotation.
  # number of pixels of arena visited (unique location, ie. visited twice counts as 1)
  # particle's x coordinate at the end
  # particle's y coordinate at the end
def simulate(x0,y0,orientation0,step):
  global x_particle,y_particle,absolute_orientation,trace

  clear_arena()
  clear_trace()

  x_particle, y_particle = x0, y0
  absolute_orientation = orientation0
  sensors = get_sensors(x_particle, y_particle,absolute_orientation)
  collision_array = place_particle(x_particle, y_particle,absolute_orientation)

  if collision_array is not None:
    print("[ERROR] initial location triggers collision.")
    sys.exit()

  iteration = 0
  log_sum_of_translation = 0 # logging
  log_sum_of_rotation = 0 # logging

  while iteration < max_iterations:
      if verbose_minimal_progress == True and iteration%gap_between_display_minimal_progress == 0:
        print ("### iteration",iteration,"/",max_iterations,"###")

      translation_per_step,rotation_per_step = step(sensors)

      log_sum_of_rotation += abs((max(-1.,min(+1.,rotation_per_step)))) # logging

      erase_particle(x_particle, y_particle)
      clean_sensors(x_particle, y_particle,absolute_orientation)
      backup_x_particle, backup_y_particle = x_particle, y_particle
      x_particle, y_particle, absolute_orientation = update_particle_position(x_particle, y_particle, absolute_orientation, (max(-1.,min(+1.,translation_per_step)))*max_translation_per_step, (max(-1.,min(+1.,rotation_per_step)))*max_rotation_per_step, iteration ,noiseFlag=False)
      sensors = get_sensors(x_particle, y_particle,absolute_orientation)
      collision_array = place_particle(x_particle, y_particle,absolute_orientation)
      if collision_array is not None: # collision occured
          if verbose == True:
              print("Collision detected!")
          if display_collision:
              print(collision_array)
          clean_sensors(x_particle, y_particle,absolute_orientation)
          x_particle, y_particle = backup_x_particle, backup_y_particle
          sensors = get_sensors(x_particle, y_particle,absolute_orientation)
          if display_collision:
              draw(collision_array)  # Draw the collision array
          absolute_orientation = update_particle_dynamics(x_particle, y_particle, absolute_orientation, (max(-1.,min(+1.,translation_per_step)))*max_translation_per_step, (max(-1.,min(+1.,rotation_per_step)))*max_rotation_per_step, iteration, collision_array)
          place_particle(x_particle, y_particle,absolute_orientation)
          if verbose:
              print("Backtrack")

      if x_particle != backup_x_particle or y_particle != backup_y_particle:
        log_sum_of_translation += math.sqrt( ( x_particle - backup_x_particle )**2 + ( y_particle - backup_y_particle)**2 )  # logging

      if display_arena == True:
        draw(arena)  # Draw the arena at each step
      if verbose == True:
        print(arena)
      iteration = iteration + 1

  log_pixels_visited = np.sum(trace == 2)

  retValues = {
      "translations": log_sum_of_translation/max_iterations,
      "rotations": log_sum_of_rotation/max_iterations,
      "coverage": log_pixels_visited/(arena_size**2),
      "x_pos_end": x_particle,
      "y_pos_end": y_particle,
      "theta_end": absolute_orientation
  }

  return retValues


#### #### ####

print("\n",date.today(), datetime.now().strftime("%H:%M:%S"),"GMT") # timestamp is greenwich time
print("OK.")


 2024-03-14 09:50:41 GMT
OK.


# Fonctions utiles (construction d'arènes)

Exécutez. Vous pouvez étudier, voire étendre le contenu.

Remarque: create_arena_walls() doit être appelé systématiquement lors de la création d'une nouvelle arène.


In [ ]:

def create_arena_walls(): # arena walls are *mandatory* to enclose the robot into a closed arena (otherwise: code will fail with out-of-bounds at some point)
  global arena_size
  create_wall(0, 0, arena_size, 1)
  create_wall(0, arena_size-1, arena_size, arena_size)
  create_wall(0, 0, 1, arena_size)
  create_wall(arena_size-1, 0, arena_size, arena_size)

def create_wall_1():
  global arena_size
  create_wall(arena_size//2+13, arena_size//2-10, arena_size//2+16, arena_size//2+10)

def create_wall_2(y_shift=0):
  global arena_size
  create_wall(arena_size//2+13, arena_size//2-10+y_shift, arena_size//2+16, arena_size//2+10+y_shift)
  create_wall(arena_size//2-16, arena_size//2-10+y_shift, arena_size//2-13, arena_size//2+10+y_shift)
  create_wall(arena_size//2-13, arena_size//2-10+y_shift, arena_size//2+13, arena_size//2-7+y_shift)
  create_wall(arena_size//2-13, arena_size//2+7+y_shift, arena_size//2-2, arena_size//2+10+y_shift)
  create_wall(arena_size//2+3, arena_size//2+7+y_shift, arena_size//2+13, arena_size//2+10+y_shift)

#### #### ####

print("\n",date.today(), datetime.now().strftime("%H:%M:%S"),"GMT") # timestamp is greenwich time
print("OK.")


 2024-03-14 09:50:41 GMT
OK.


# Initialisation des paramètres expérimentaux et des options recommandées d'affichage de messages

In [ ]:
# experimental settings

arena_size = 100
max_translation_per_step = 1.0 # >0. pixel per move.
max_rotation_per_step = 10.0 # >0, in degrees
max_iterations = 5001 # nb of simulator iterations for evaluating one robot

# default values for display debug (may be changed later)

verbose_minimal_progress = True # suggested: True -- useful. Tune gap_between_... to limit the nb of messages
gap_between_display_minimal_progress = int(max_iterations/5) # used if verbose_minimal_progress = True
verbose = False # suggested: False -- used for debug
display_collision = False # suggested: False -- used for debug
display_arena = False # suggested: False -- if True, display arena for *all* iterations
display_trace = True # suggested: True
display_cast = False # suggested: False -- if True, display sensor ray casting for *all* iterations (even w/o display)

#### #### ####

print("\n",date.today(), datetime.now().strftime("%H:%M:%S"),"GMT") # timestamp is greenwich time
print("OK.")


 2024-03-14 09:50:44 GMT
OK.


---
---
---

#<font color="orange">Partie 1. Prise en main : stratégies comportementales</font>

Le robot dispose de 8 senseurs (tous ne seront pas utilisés dans la suite) et de deux commandes motrices (translation et rotation). Toutes les commandes sont normalisées (senseurs dans [0,1], translation et rotation dans [-1,+1]).

Les dimensions du robot sont de 2x2 unités. Une unité correspond à un pixel. Cependant, même si l'affichage est discret, le robot se déplace de manière continue dans l'environnement (position/orientation sont des valeurs réelles). La gestion des collisions est faîtes de manière discrète, les déplacements et la mise à jour des valeurs senseurs est faite sur la base de valeurs continues.

La structure des senseurs est la suivante:
  - [0] avant
  - [1] avant-gauche
  - [2] gauche
  - [3] arrière-gauche
  - [4] arrière
  - [5] arrière-droite
  - [6] droite
  - [7] avant-droite

L'orientation est définie comme suit:
  - *absolute_orientation* est utilisé à l'initialisation pour positionner le robot. la valeur 0 indique que le robot pointe vers la droite de l'arène. =-90 vers le bas. etc. (sens trigonométrique)
  - *rotation* est une commande motrice du robot. Elle s'exprime dans le sens trigonométrique (le robot tourne à gauche si inférieure à 0). Elle est définie entre -1 et +1. Cependant, la rotation effective dépend du paramètre * max_rotation_per_step* (non modifiable), qui donne l'angle de rotation maximum possible entre deux itérations du simulateur.
  - exemple: une rotation de -1.0 sachant que max_rotation_per_step=10 provoque une rotation dans le sens inverse des aiguilles d'une montre de 10° par itération.

La translation est définie comme suit:
  - entre -1 (vitesse max. en arrière) et 1 (vitesse max. en avant). pas de déplacement si la translation vaut 0.
  - la translation effective dépend du paramètre *max_translation_per_step* (non modifiable) qui donne le nombre d'unité de déplacement maximum par itération de l'algorithme.
  - exemple: une translation de -0.5 sachant que max_translation_per_step=10 provoque une translation vers l'arrière de 5 unités par itération.
  - Remarque: la notion d'avant et d'arrière est arbitraire. En pratique, cette asymétrie est une interprétation que nous faisons en observant le comportement. En particulier, le résultat d'une optimisation peut *vous* donner l'*impression* que le robot se déplace en reculant (cf. suite du sujet), mais cette observation n'a pas de sens du point de vue du robot (en particulier si les senseurs sont disposés de manière symétrique, ce qui est le cas ici).

<font color="red">EXERCICES:</font>

- Etudiez et exécutez les trois exemples de stratégies comportementales. Chaque évaluation produit un certain nombre d'informations: des informations quantitatives sur la trajectoire (cf. retour de la fonction) et deux graphes permettant d'observer (a) la trajectoire du robot et (b) sa position/orientation finale.

- Etudiez et exécutez le code de la dernière cellule. Vous observerez qu'en modifiant la valeur de *display_arena* pour la mettre à True, on affichera image par image l'exécution de la simulation. Cela pourra vous être utile par la suite pour comprendre par observation un comportement (n'oubliez pas de remettre la variable à False ensuite, afin de ne pas ralentir inutilement l'exécution du code).

# Un robot peu prudent

In [ ]:
# policy function

def my_controller_dumb(sensors):
  translation = 1.0
  rotation = 0.
  return translation,rotation

# Main

init()

print ("#\n# Policy no.1 -- dumb robot\n#")

display_arena=False # diplay step-by-step if True.
verbose_minimal_progress = True # suggested: True -- useful. Tune gap_between_... to limit the nb of messages

#clear_arena()
create_arena_walls()
create_wall_1()

x_init = arena_size//2 - particle_box/2
y_init = arena_size//2 - particle_box/2
theta_init = 0

retValues = simulate(x_init,y_init,theta_init,my_controller_dumb)

print ()
print ("Statistiques:")
print ("\t translations      :", retValues['translations']) # Somme des valeurs absolues de translations. valeurs normalisées entre 0 et 1.
print ("\t rotations         :", retValues['rotations']) # Somme des valeurs absolues de rotations. Valeur normalisée entre 0 et 1.
print ("\t couverture        :", retValues['coverage']) # taux de couverture. Valeur normalisée entre 0 et 1 (arène entièrement couverte).
print ("\t position initiale : (",x_init,",",y_init,",",theta_init,")" ) # init: x,y,\theta
print ("\t position finale   : (",retValues['x_pos_end'],",",retValues['y_pos_end'],",",retValues['theta_end'],")" ) # final: init: x,y,\theta
print ()

print ("# Final state")
show_arena()

print ("# Trajectory")
show_trace()


# Un robot prudent mais fébrile

Le robot s'arrête s'il voit un mur, et effectue tout le temps une rotation au hasard (ce qui lui permet parfois de se réorienter suffisamment pour repartir).

In [ ]:
def my_controller_cautious_random(sensors):
  translation = 0.0
  if sensors[0] == 1.0: # front sensor (in [0,+1], +1 means no obstacle detected)
    translation = 1.0 # max translation speed
  else:
    translation = 0.0 # no translation
  rotation = (random.random()-0.5)*2. # rotational noise with max. amplitude (result in -1,+1)
  return translation,rotation

# Main

init()

print ("#\n# Policy no.2 -- cautious-random robot\n#")

display_arena=False # diplay step-by-step if True.
verbose_minimal_progress = True # suggested: True -- useful. Tune gap_between_... to limit the nb of messages
display_trace = True

#clear_arena()
create_arena_walls()
create_wall_1()

x_init = arena_size//2 - particle_box/2
y_init = arena_size//2 - particle_box/2
theta_init = 0

retValues = simulate(x_init,y_init,theta_init,my_controller_cautious_random)

print ()
print ("Statistiques:")
print ("\t translations      :", retValues['translations']) # Somme des valeurs absolues de translations. valeurs normalisées entre 0 et 1.
print ("\t rotations         :", retValues['rotations']) # Somme des valeurs absolues de rotations. Valeur normalisée entre 0 et 1.
print ("\t couverture        :", retValues['coverage']) # taux de couverture. Valeur normalisée entre 0 et 1 (arène entièrement couverte).
print ("\t position initiale : (",x_init,",",y_init,",",theta_init,")" ) # init: x,y,\theta
print ("\t position finale   : (",retValues['x_pos_end'],",",retValues['y_pos_end'],",",retValues['theta_end'],")" ) # final: init: x,y,\theta
print ()

print ("# Final state")
show_arena()

print ("# Trajectory")
show_trace()

# Un robot éviteur d'obstacles

In [ ]:
def my_controller_avoider(sensors):
  translation = sensors[0]
  rotation = 1.0*sensors[1] - 1.0*sensors[-1] + (random.random()-0.5)*0.1
  #print (sensors,translation,rotation,absolute_orientation)
  return translation,rotation

# Main

init()

print ("#\n# Policy no.3 -- obstacle-avoider robot\n#")

display_arena=False # diplay step-by-step if True.
verbose_minimal_progress = True # suggested: True -- useful. Tune gap_between_... to limit the nb of messages
display_trace = True

#clear_arena()
create_arena_walls()
create_wall_1()

x_init = arena_size//2 - particle_box/2
y_init = arena_size//2 - particle_box/2
theta_init = 0

retValues = simulate(x_init,y_init,theta_init,my_controller_avoider)

print ()
print ("Statistiques:")
print ("\t translations      :", retValues['translations']) # Somme des valeurs absolues de translations. valeurs normalisées entre 0 et 1.
print ("\t rotations         :", retValues['rotations']) # Somme des valeurs absolues de rotations. Valeur normalisée entre 0 et 1.
print ("\t couverture        :", retValues['coverage']) # taux de couverture. Valeur normalisée entre 0 et 1 (arène entièrement couverte).
print ("\t position initiale : (",x_init,",",y_init,",",theta_init,")" ) # init: x,y,\theta
print ("\t position finale   : (",retValues['x_pos_end'],",",retValues['y_pos_end'],",",retValues['theta_end'],")" ) # final: init: x,y,\theta
print ()

print ("# Final state")
show_arena()

print ("# Trajectory")
show_trace()

# Affichage pas à pas de la simulation

Utile pour étudier la trajectoire d'un robot, mais très lent. A utiliser avec extrême parcimonie.

Exemple avec le "robot prudent mais fébrile".

In [ ]:

# initial conditions

x_init = arena_size//2 - particle_box/2
y_init = arena_size//2 - particle_box/2
theta_init = 0

display_trace = True
display_arena = True

init() # perform only once
create_arena_walls()
create_wall_1()

retValues = simulate(x_init,y_init,theta_init,my_controller_cautious_random)
show_trace()
show_arena()

print ()
print ("Statistiques:")
print ("\t translations      :", retValues['translations']) # Somme des valeurs absolues de translations. valeurs normalisées entre 0 et 1.
print ("\t rotations         :", retValues['rotations']) # Somme des valeurs absolues de rotations. Valeur normalisée entre 0 et 1.
print ("\t couverture        :", retValues['coverage']) # taux de couverture. Valeur normalisée entre 0 et 1 (arène entièrement couverte).
print ("\t position initiale : (",x_init,",",y_init,",",theta_init,")" ) # init: x,y,\theta
print ("\t position finale   : (",retValues['x_pos_end'],",",retValues['y_pos_end'],",",retValues['theta_end'],")" ) # final: init: x,y,\theta
print ()

---
---
---

#<font color="orange">Partie 2. Prise en main : exploration de l'espace des politiques</font>

Nous allons utiliser le code précédent pour générer au hasard des politiques, en variant les paramètres d'une fonction simple de contrôle. La politique du robot est dirigée par une combinaison linéaire de quatre entrées sensorielles. Les poids appliqués sur les entrées sont tirés au hasard entre chaque essai. La variable globale *weights* est utilisée pour stocker les paramètres de la politique.

<font color="red">EXERCICE :</font>

- Exécutez et étudiez l'exemple de recherche au hasard. Observez en particulier comme la politique est modifiée entre chaque essai. Observez aussi la direction préférentielle du robot (vers l'avant ou vers l'arrière?).

- modifiez le code de la recherche au hasard pour calculer la fitness de "*déplacement*" vu en cours (maximiser la vitesse de translation, minimiser la vitesse de rotation -- on oublie les senseurs!) en utilisant les données  retournées par une simulation. Vous devrez (1) écrire la fonction fitness qui lance une simulation et renvoie la performance et (2) conserver le meilleur individu. Vous ferez 200 essais.

- rejouez le meilleur individu en affichant uniquement la trajectoire complète, la position finale, la performance donnée par la fitness et les données produites par la simulation (translations, rotations, couverture, position x/y à la fin).

In [ ]:

def my_controller_NN_4sensors(sensors):
  global weights
  translation = weights[0]*sensors[-2] + weights[1]*sensors[0] + weights[2]*sensors[2] + weights[3]*sensors[4] + weights[4]
  rotation = weights[5]*sensors[-2] + weights[6]*sensors[0] + weights[7]*sensors[2] + weights[8]*sensors[4] + weights[9]
  #print (sensors,translation,rotation,absolute_orientation)
  return translation,rotation

print ("#\n# random exploration of the policy space using a linear weighted combination of sensory inputs#")

display_arena=False # diplay step-by-step if True.
display_trace = True

init()

#clear_arena()
create_arena_walls()
create_wall_2(-20)

x_init = arena_size//2 - particle_box/2
y_init = arena_size//2 - particle_box/2
theta_init = 0

nb_runs = 20
max_iterations = 501

for i in range(nb_runs):
  print ("run no.",(i+1),"of",nb_runs)
  weights = np.random.uniform(-1.0, 1.0, size=10).tolist()
  retValues = simulate(x_init,y_init,0,my_controller_NN_4sensors)
  print (retValues)
  show_trace()

print ("Terminé.")


---
---
---

#<font color="orange">Partie 3. Evolution</font>

A partir du code précédent, nous allons maintenant implémenter deux algorithmes d'évolution artificielle vue lors de la séance précédente: (1+1)-ES et CMAES (n'hésitez pas à faire des copier-coller). Pour ne pas alourdir l'affichage, vous pouvez vous contenter d'afficher uniquement la valeur de la fitness pour chaque individu évalué pendant l'évolution.

<font color="red">EXERCICES :</font>

- en utilisant le code que vous avez réalisé lors du dernier TP, implémentez l'algorithme (1+1)-ES (avec la règle des 1/5e si vous l'avez fait) pour optimiser la fitness de déplacement précédemment implémentée. Lancez l'algorithme sur 200 itérations, affichez le meilleur résultat (performance et graphes).

- Idem avec CMAES. Pour limiter le budget d'évaluations de CMAES vous pouvez utiliser la commande *es.opts.set({'maxfevals': mon_budget_en_evaluations})*. A noter que CMAES s'arrétera *à la fin* de la première génération qui aura épuisée le budget d'évaluations, ie. le nombre d'évaluations effective peut dépasser la limite prévue par le budget -- on considère que c'est négligeable ici.

- Comparez (1+1)-ES et CMAES en générant les courbes montrant l'évolution des best-evers au cours du temps pour chaque algorithmes.

# (1+1)-ES 1/5th rule


In [ ]:

def my_controller_NN_4sensors(sensors):
  global weights
  translation = weights[0]*sensors[-2] + weights[1]*sensors[0] + weights[2]*sensors[2] + weights[3]*sensors[4] + weights[4]
  rotation = weights[5]*sensors[-2] + weights[6]*sensors[0] + weights[7]*sensors[2] + weights[8]*sensors[4] + weights[9]
  #print (sensors,translation,rotation,absolute_orientation)
  return translation,rotation

def launch_oneplusone_withOneFifthRule(individual, sigma_init, nbeval, display, ma_func):
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  return besteverIt,besteverFit,besteverSolution,data

def my_fitness_func_maximize(params): # fitness function
  global x_init,y_init,theta_init,weights
  weights = params
  retValues = simulate(x_init,y_init,theta_init,my_controller_NN_4sensors)
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  fitness = ...
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  if display_trace == True:
      show_trace()
  return fitness

# Main

print ("#\n# (1+1)-ES 1/5th rule for maximising translation and minimasing rotation#")

display_arena=False # diplay step-by-step if True.
verbose_minimal_progress = False # suggested: True -- useful. Tune gap_between_... to limit the nb of messages
display_trace = False

init()

#clear_arena()
create_arena_walls()
create_wall_2(-20)

max_iterations = 501 # nb of simulation iterations for one robot
max_iterations_evolve = 201 # nb of evolution iterations

x_init = arena_size//2 - particle_box/2
y_init = arena_size//2 - particle_box/2
theta_init = 0

initial_solution = np.random.uniform(-1.0, 1.0, size=10).tolist()
besteverIt,besteverFit,besteverSolution,data = launch_oneplusone_withOneFifthRule(initial_solution,nbeval=max_iterations_evolve,sigma_init=0.0001,display=True,ma_func=my_fitness_func_maximize)

print ("\nRejoue et affiche la meilleure solution")

weights = np.copy(besteverSolution).tolist()

retValues = simulate(x_init,y_init,theta_init,my_controller_NN_4sensors)

print (retValues)
print ("# Final state")
show_arena()
print ("# Trajectory")
show_trace()
print ("Terminé.")


# CMAES



In [ ]:

def my_controller_NN_4sensors(sensors):
  global weights
  translation = weights[0]*sensors[-2] + weights[1]*sensors[0] + weights[2]*sensors[2] + weights[3]*sensors[4] + weights[4]
  rotation = weights[5]*sensors[-2] + weights[6]*sensors[0] + weights[7]*sensors[2] + weights[8]*sensors[4] + weights[9]
  #print (sensors,translation,rotation,absolute_orientation)
  return translation,rotation

def launch_cmaes(dim, display, ma_func, max_evaluations):
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  return es.result.fbest,es.result.xbest

def my_fitness_func_for_cmaes(params):
  global x_init,y_init,theta_init,weights,eval
  weights = params
  retValues = simulate(x_init,y_init,theta_init,my_controller_NN_4sensors)
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  fitness = ...
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  # !!!!!! !!!!!! !!!!!! !!!!!! !!!!!! A COMPLETER !!!!!! !!!!!! !!!!!! !!!!!! !!!!!!
  if display_trace == True:
      show_trace()
  print (eval,":",fitness)
  eval += 1
  return fitness

# display debug options

display_arena=False # step-by-step display.
verbose_minimal_progress = False # suggested: True -- useful. Tune gap_between_... to limit the nb of messages
display_trace = False

# experimental settings

arena_size = 100
max_translation_per_step = 1.0 # >0. pixel per move.
max_rotation_per_step = 10.0 # >0, in degrees
max_iterations = 501 # nb of simulator iterations for evaluating one robot
max_iterations_evolve = 201 # nb of evolution iterations

# initial conditions

x_init = arena_size//2 - particle_box/2
y_init = arena_size//2 - particle_box/2
theta_init = 0

init() # perform only once
create_arena_walls()
create_wall_2(-20)

print ("\n=-=-=-=-=\n=-=-=-=-= Robot with CMA-ES\n=-=-=-=-=")

eval = 0
besteverFitness,besteverParams = launch_cmaes(dim=10,display=False,ma_func=my_fitness_func_for_cmaes, max_evaluations=max_iterations_evolve)

print ("besteverFitness",besteverFitness)
print ("besteverParams",besteverParams)

retValues = my_fitness_func_for_cmaes(besteverParams)

print (retValues)
print ("# Final state")
show_arena()
print ("# Trajectory")
show_trace()
print ("Terminé.")


# Replay

A toutes fins utiles, exécutez cette cellule pour obtenir un replay (avec affichage pas à pas si display_arena=True) de la trajectoire du robot avec les meilleurs paramètres. Veillez à ce que *besteverParams* contienne bien les paramètres voulus (ce qui est le cas si vous venez d'exécutez la cellule précédente).

In [ ]:

# initial conditions

x_init = arena_size//2 - particle_box/2
y_init = arena_size//2 - particle_box/2
theta_init = 0

display_trace = True
display_arena = False

init() # perform only once
create_arena_walls()
create_wall_2(-20)

print ("besteverFitness",besteverFitness)
print ("besteverParams",besteverParams)

weights = besteverParams
retValues = simulate(x_init,y_init,theta_init,my_controller_NN_4sensors)
show_trace()
show_arena()

print ()
print ("Statistiques:")
print ("\t translations      :", retValues['translations']) # Somme des valeurs absolues de translations. valeurs normalisées entre 0 et 1.
print ("\t rotations         :", retValues['rotations']) # Somme des valeurs absolues de rotations. Valeur normalisée entre 0 et 1.
print ("\t couverture        :", retValues['coverage']) # taux de couverture. Valeur normalisée entre 0 et 1 (arène entièrement couverte).
print ("\t position initiale : (",x_init,",",y_init,",",theta_init,")" ) # init: x,y,\theta
print ("\t position finale   : (",retValues['x_pos_end'],",",retValues['y_pos_end'],",",retValues['theta_end'],")" ) # final: init: x,y,\theta
print ()

---
---
---

#<font color="orange">Partie 4. Robustesse et généralisation</font>

Nous avons jusqu'ici utilisé les mêmes valeurs pour la position et l'orientation initiale. Cependant cela peut provoquer un surapprentissage dépendant de ces conditions initiales. C'est ce que nous allons étudier ici.

<font color="red">EXERCICES :</font>

- Testez le résultat obtenu précédemment par CMAES (ie. sans relancer l'optimisation) en utilisant *create_wall_2()* à la place de *create_wall_2(-20)* pour construire l'environnement. Exécutez et commentez le résultat.

- Relancez l'optimisation avec CMAES en utilisant *create_wall_2()* comme condition initiale. Commentez le résultat.

- Modifier l'évaluation d'un individu pour choisir aléatoire le point de départ du robot: soit au centre de l'arène, soit au centre de la mini-arène définie par *create_wall_2(-20)*. Qu'observez-vous? Commentez. en particulier: comment faire mieux dans le cas général?



In [ ]:
# A compléter

---
---
---

#<font color="orange">Partie 5. Fonctions objectifs et tâches</font>

A partir de cet exercice, vous pouvez augmentez (si vous le souhaitez) le budget d'évaluations utilisé. Pour cet exercice, on utilisera *create_wall_1* pour construire l'environnement initial. La position initiale du robot sera au centre de l'arène et son orientation sera de zéro (comme au début).

<font color="red">EXERCICES :</font>

- Vous remarquerez que la fonction simulation retourne un score de couverture. Avec CMAES, optimisez la politique pour maximiser ce score de couverture en l'utilisant comme valeur de fitness.

- Le score de couverture est une bonne mesure de la qualité d'une stratégie existante, mais pas forcément un bon guide pour l'optimisation. Ecrivez une fonction maximisant la couverture, c'est-à-dire qui ne contiendra pas forcément le score de couverture, mais aussi d'autres éléments pouvant guider l'optimisation, combiné ou non avec le score de couverture. Vous comparerez avec les résultats obtenus avec le score de couverture originel et ceux obtenus avec votre fonction fitness. Vous utiliserez le score de couverture pour tracer la performance des best-ever au cours de l'évolution dans les deux cas, afin que les résultats soient interprétables.



In [ ]:
# A compléter

---
---
---

#<font color="orange">Partie 6. Architectures neuronales</font>

Nous allons maintenant modifier la structure de la politique en implémentant des architectures de réseaux de neurones artificiels classiques (mais relativement "légères"). A part la première question, il s'agit d'un exercice difficile à faire *uniquement* si vous avez terminé (et réussi) tout ce qui précède.

En particulier, la dernière question correspond à la mise en place d'une méthodologie rigoureuse de test. Par les temps de calcul qu'elle implique, elle *ne peut pas* être réalisée dans le temps de ce TP. Elle est énoncée ici à titre indicatif (ie. il ne s'agit pas d'un devoir à la maison).

<font color="red">EXERCICES :</font>

- Modifiez la structure de la politique, qui est actuellement une combinaison linéaire des entrées pondérées par des poids. Vous implémenterez un Perceptron simple (fonction d'activation non-linéaire)

- idem avec un perceptron multi-couche (nombre de noeuds à discretion, par exemple 3)

- Comparez ces architectures (y compris celle par défaut). Vous testerez sur les deux fonctions fitness utilisées précédemment: déplacement et couverture. Pour l'instant on se contentera d'un seul run par condition expérimentale (ie. tâche et architecture du réseau).

- idem avec un réseau Elman, c'est à dire avec des connexions récurrentes pour les noeuds de la couche cachée. Chaque noeud de la couche cachée projette sur tous les autres et sur lui-même.

- idem avec un réseau Jordan, c'est à dire avec deux sorties supplémentaires qui projettent sur deux entrées supplémentaires (ie. connexions récurrentes en réinjectant la valeur de certaines sorties produites à *t-1* en valeurs d'entrées à *t*)

- Comparez toutes ces architectures (y compris celle par défaut) avec plusieurs runs indépendant pour chaque conditions expérimentales (type de réseaux, tâches) en variant la position de départ dans un cercle de 4 unités ainsi que l'orientation à chaque évaluation (il sera peut-être nécessaire de ré-évaluer chaque individu pour avoir une bonne approximation de sa performance moyenne). Pour chaque condition expérimentale, vous pouvez vous contentez d'afficher les données brutes du best-ever pour chaque run fait (ou un boxplot si vous avez 11 runs par setup).


In [ ]:
# A compléter

---
---
---

#<font color="orange">Fin du sujet.</font>

---
---
---
